# Notebook 2: Feature Engineering
Goal: Build the complete feature matrix `features_long.parquet` from all 7 datasets.

In [ ]:
import sys
import os
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

PROCESSED_DIR = '../data/processed/'
print('Environment ready.')

## 1. Run Feature Pipeline

In [ ]:
from src.data.feature_pipeline import build_features

# build_features() reads all 7 raw CSVs, normalises country names,
# merges on (origin, dest, year, product_code), computes lag/delta
# features, and writes features_long.parquet to data/processed/.
df = build_features(save=True)

print(f'Feature matrix shape: {df.shape}')
print(f'\nDtypes:')
print(df.dtypes.to_string())
print(f'\nSample rows:')
display(df.head(5))

## 2. Feature Distribution Analysis

In [ ]:
# Key numeric features to inspect
PLOT_FEATURES = ['freight_rate', 'bilateral_lsci', 'origin_lsci', 'origin_teu']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, feat in zip(axes, PLOT_FEATURES):
    data = df[feat].dropna()
    ax.hist(data, bins=50, edgecolor='white', color='steelblue', alpha=0.85)
    ax.set_title(feat)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}') if data.max() > 1000 else mticker.ScalarFormatter())
    skew_val = float(data.skew())
    ax.text(0.97, 0.95, f'skew={skew_val:.2f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=9, color='darkred')

import matplotlib.ticker as mticker  # ensure import
fig.suptitle('Feature Distributions (observed rows)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3. Correlation Analysis

In [ ]:
ML_FEATURES = [
    'bilateral_lsci', 'origin_lsci', 'dest_lsci',
    'origin_teu', 'dest_teu',
    'origin_fleet_dwt', 'dest_fleet_dwt',
    'origin_port_calls', 'dest_port_calls',
    'trade_value_usd',
    'year_numeric', 'is_covid',
    'freight_rate_lag1', 'freight_rate_delta1',
]

# Use only observed (non-null) freight_rate rows
obs = df[df['freight_rate'].notna()].copy()

# Keep only features that exist in the dataframe
avail_feats = [f for f in ML_FEATURES if f in obs.columns]
corr_cols = avail_feats + ['freight_rate']

corr_matrix = obs[corr_cols].corr()

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.3,
    ax=ax
)
ax.set_title('Pearson Correlation Matrix — ML Features vs Freight Rate')
plt.tight_layout()
plt.show()

# Top-5 features correlated with freight_rate
top5 = (
    corr_matrix['freight_rate']
    .drop('freight_rate')
    .abs()
    .sort_values(ascending=False)
    .head(5)
)
print('Top 5 features correlated with freight_rate:')
print(top5.to_string())

## 4. COVID-19 Signal

In [ ]:
yearly = (
    obs
    .groupby('year')['freight_rate']
    .agg(['mean', 'std'])
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(yearly['year'].astype(str), yearly['mean'], color='steelblue', alpha=0.8, label='Mean')
ax.errorbar(
    yearly['year'].astype(str), yearly['mean'],
    yerr=yearly['std'], fmt='none', color='black', capsize=4, linewidth=1.2
)
ax.axvspan(
    yearly['year'].astype(str).tolist().index('2020') - 0.5,
    yearly['year'].astype(str).tolist().index('2021') + 0.5,
    alpha=0.12, color='red', label='COVID-19 shock'
)
ax.set_xlabel('Year')
ax.set_ylabel('Mean Freight Rate (USD/TEU)')
ax.set_title('Annual Mean Freight Rate ± 1 Std Dev')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

for yr in [2019, 2020, 2021, 2022]:
    row = yearly[yearly['year'] == yr]
    if not row.empty:
        print(f'  {yr}: mean = {row["mean"].values[0]:,.0f} USD/TEU')

## 5. Missing Value Summary

In [ ]:
null_pct = df[avail_feats + ['freight_rate']].isnull().mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(10, 5))
null_pct.plot.barh(ax=ax, color='coral', edgecolor='white')
ax.set_xlabel('% Null')
ax.set_title('Null Rate per Feature Column (after pipeline merge)')
ax.axvline(36, color='steelblue', linestyle='--', label='36% threshold')
ax.legend()
plt.tight_layout()
plt.show()

print(null_pct.to_string())

## 6. Feature Engineering Summary

| Feature | Source Dataset | Description |
|---|---|---|
| `freight_rate` | transport_cost_by_product | Target variable — USD per TEU for (origin, dest, product, year) |
| `bilateral_lsci` | bilateral_shipping_connectivity_index | UNCTAD bilateral LSCI score between origin and destination economy |
| `origin_lsci` | liner_shipping_connectivity_index | Country-level LSCI for origin economy |
| `dest_lsci` | liner_shipping_connectivity_index | Country-level LSCI for destination economy |
| `origin_teu` | container_port_throughput | Annual TEU throughput of origin country's largest port |
| `dest_teu` | container_port_throughput | Annual TEU throughput of destination country's largest port |
| `origin_fleet_dwt` | merchant_fleet_ownership | Total fleet DWT registered to origin economy |
| `dest_fleet_dwt` | merchant_fleet_ownership | Total fleet DWT registered to destination economy |
| `origin_port_calls` | port_calls_and_performance | Number of container ship port calls at origin country |
| `dest_port_calls` | port_calls_and_performance | Number of container ship port calls at destination country |
| `trade_value_usd` | trade_value_by_partner | Bilateral merchandise trade value (USD) for the corridor |
| `year_numeric` | Derived | Integer year, used as a trend feature |
| `is_covid` | Derived | Binary flag: 1 for 2020–2021, 0 otherwise |
| `freight_rate_lag1` | Derived | Freight rate for same corridor, previous year |
| `freight_rate_delta1` | Derived | Year-on-year change in freight rate for the corridor |